In [17]:
import pandas as pd
import unicodedata
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from openpyxl import load_workbook

# 1. Función de limpieza rápida
def normalizar_texto(texto):
    if pd.isna(texto): return ""
    s = str(texto).strip().upper()
    return unicodedata.normalize('NFKD', s).encode('ASCII', 'ignore').decode('utf-8')

# 2. Función optimizada: Lectura parcial (solo filas 22 a 25)
def procesar_archivo_rapido(file_path):
    lista_hojas = []
    # Abrimos el archivo en modo "solo lectura" para obtener metadata rápido
    try:
        # Esto es mucho más rápido que cargar todo el contenido
        wb = load_workbook(file_path, read_only=True, data_only=False)
        
        for sheet_name in wb.sheetnames:
            sheet = wb[sheet_name]
            
            # FILTRO: Ignorar ocultas y administrativas
            if sheet.sheet_state != 'visible' or sheet_name.upper() in {"ADITIVOS", "GENERAL", "LISTAS", "TIEMPOS"}:
                continue
            
            # LEER SOLO LAS FILAS NECESARIAS (22 a 25) usando pandas directamente
            # skiprows=22, nrows=3 nos trae Fila 23, 24 y 25
            df_slice = pd.read_excel(file_path, sheet_name=sheet_name, skiprows=22, nrows=3, header=None)
            
            t23 = df_slice.iloc[0].tolist() # Fila 23
            t24 = df_slice.iloc[1].tolist() # Fila 24
            
            # Logica de consolidacion
            t23_rellenos = []
            ultimo_val = "XP"
            for val in t23:
                txt = normalizar_texto(val)
                if txt != "":
                    ultimo_val = txt
                    t23_rellenos.append(ultimo_val)
                else:
                    t23_rellenos.append(ultimo_val)
            
            headers = []
            for t1, t2 in zip(t23_rellenos, t24):
                sub_t2 = normalizar_texto(t2)
                if t1 == "XP": head = sub_t2
                elif sub_t2 == "": head = t1
                else: head = f"{t1}_{sub_t2}"
                headers.append(head)
            
            if len(headers) > 0: headers[0] = "FECHA"
            
            lista_hojas.append({
                "ID": f"{file_path.name[:15]}_{sheet_name}",
                "Headers": headers
            })
    except Exception as e:
        print(f"Error crítico en {file_path.name}: {e}")
    return lista_hojas

# 3. EJECUCIÓN PARALELA (Aquí está la velocidad)
BASE_DIR = Path(r"C:\Proyectos Python\Detallados\archivos")
archivos = list(BASE_DIR.glob("*.xlsx"))

inventario = []
with ThreadPoolExecutor(max_workers=4) as executor: # Ajusta max_workers según tu CPU
    resultados = list(executor.map(procesar_archivo_rapido, archivos))
    for res in resultados:
        inventario.extend(res)

# 4. Construir Matriz (Moda)
max_cols = max(len(h["Headers"]) for h in inventario)
matriz_data = {}
for item in inventario:
    cols = item["Headers"] + [None] * (max_cols - len(item["Headers"]))
    matriz_data[item["ID"]] = cols

df_matrix = pd.DataFrame(matriz_data)
estandar = df_matrix.mode(axis=1)[0] # La moda por posición

# 5. Exportar con Formato Condicional (Rojo = Discrepancia)
output_path = BASE_DIR.parent / "Auditoria_Visual_Discrepancias.xlsx"
writer = pd.ExcelWriter(output_path, engine='xlsxwriter')
df_matrix.to_excel(writer, sheet_name='Matriz', index=False)

workbook = writer.book
worksheet = writer.sheets['Matriz']
red_format = workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006'})

# Formato condicional aplicado sobre toda la matriz
for r in range(len(df_matrix)):
    moda_val = estandar[r]
    # Compara contra la moda de esa fila
    worksheet.conditional_format(1, 0, len(df_matrix.columns), len(df_matrix.columns) - 1, {
        'type': 'expression',
        'criteria': f'A1<>"{moda_val}"', # Ajusta según la celda, xlsxwriter es sensible
        'format': red_format
    })

writer.close()
print(f"✅ Auditoría completada en segundos. Archivo guardado en: {output_path}")

Error crítico en RD.402.P.01.F.01 Reporte Detallado de Avance YAURICOCHA  - JULIO.xlsx: single positional indexer is out-of-bounds
✅ Auditoría completada en segundos. Archivo guardado en: C:\Proyectos Python\Detallados\Auditoria_Visual_Discrepancias.xlsx


c:\Proyectos Python\Detallados\venv\Lib\site-packages\xlsxwriter\worksheet.py:2928: UserWarning: Unknown value 'expression' for parameter 'type' in conditional_format()
  warn(
